In [ ]:
!pip install -qU transformers langchain-cohere langchain-chroma
!pip install -qU transformers torch==2.6 

# !pip install -qU torch==2.5.0    # reset it back to this version after running this demo

### Image Analysis

In [ ]:
from langchain_community.document_loaders import ImageCaptionLoader

#  load images from data folder
list_image_urls = [
    "./data/birdimg.jpg","./data/carimg.jpg"
    ]

In [ ]:
#  getting captions for images using ImageCaptionLoader() from LangChain

loader = ImageCaptionLoader(images=list_image_urls)
list_docs = loader.load()
list_docs

## Display the image

In [ ]:
import requests
from PIL import Image

Image.open(list_image_urls[0]).convert("RGB")

In [ ]:
import requests
from PIL import Image

Image.open(list_image_urls[1]).convert("RGB")

# loading image captions into vector db

In [ ]:
from langchain_chroma import Chroma
from langchain_cohere import CohereEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(list_docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=CohereEmbeddings(cohere_api_key=COHERE_API_KEY,model="embed-english-v3.0"))

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

### Prompt design and interaction with LLM for image analysis

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_cohere import ChatCohere
import warnings
warnings.filterwarnings('ignore')


model = ChatCohere(cohere_api_key=COHERE_API_KEY,temperature=0)

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(model, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)


### Run the Query 

In [ ]:
response = rag_chain.invoke({"input": "What animals are in the images?"})

print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "What kind of images are there?"})

print(response["answer"])